In [1]:
from pathlib import Path
from copy import deepcopy
import sys

import pandas as pd
import wandb

PROJECT_ROOT = Path(
    r"C:\Users\elsae\Downloads\ecommerce-purchase-prediction"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    load_raw_data,
    validate_manifest,
    get_split,
)

print("Folder proyek:", PROJECT_ROOT)

Folder proyek: C:\Users\elsae\Downloads\ecommerce-purchase-prediction


In [2]:
SOURCE_RUN_PATH = (
    "elsaelisa/ecommerce-purchase-prediction/u1b7ipvz"
)

source_run = wandb.Api().run(SOURCE_RUN_PATH)
base_config = deepcopy(dict(source_run.config))

# Membaca dataset dan pembagian data yang sudah dibuat.
df, manifest = load_raw_data(base_config["data"])
validate_manifest(df, manifest)

# Hanya mengambil bagian training untuk cross-validation.
X_cv, y_cv = get_split(
    df,
    manifest,
    split_name="train",
    target_column=base_config["data"]["target"],
    exclude_columns=base_config["features"]["exclude_columns"],
)

# Menyamakan urutan kelompok dengan urutan baris fitur.
groups_cv = (
    manifest.set_index("row_id")
    .loc[X_cv.index, "group_id"]
)

assert X_cv.index.equals(y_cv.index)
assert X_cv.index.equals(groups_cv.index)
assert not groups_cv.isna().any()

print("Model dasar:", source_run.name)
print("Ukuran fitur:", X_cv.shape)
print("Ukuran target:", y_cv.shape)
print("Jumlah kelompok:", groups_cv.nunique())

Model dasar: xgb_all_features
Ukuran fitur: (7406, 17)
Ukuran target: (7406,)
Jumlah kelompok: 7327


In [3]:
from sklearn.model_selection import StratifiedGroupKFold

# Pembagian tetap berdasarkan data dan urutan baris yang sama.
cv_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=False,
)

cv_splits = list(
    cv_splitter.split(X_cv, y_cv, groups=groups_cv)
)

fold_records = []

for fold, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
    train_groups = set(groups_cv.iloc[train_idx])
    valid_groups = set(groups_cv.iloc[valid_idx])

    shared_groups = len(train_groups & valid_groups)

    assert shared_groups == 0, (
        f"Ada kelompok yang tumpang tindih pada fold {fold}."
    )

    fold_records.append({
        "fold": fold,
        "train_sessions": len(train_idx),
        "validation_sessions": len(valid_idx),
        "train_purchase_pct": y_cv.iloc[train_idx].mean() * 100,
        "validation_purchase_pct": y_cv.iloc[valid_idx].mean() * 100,
        "shared_groups": shared_groups,
    })

fold_summary = pd.DataFrame(fold_records)

display(fold_summary.round(2))

,fold,train_sessions,validation_sessions,train_purchase_pct,validation_purchase_pct,shared_groups
0,1,5924,1482,15.97,15.99,0
1,2,5925,1481,15.98,15.94,0
2,3,5925,1481,15.98,15.94,0
3,4,5925,1481,15.97,16.00,0
4,5,5925,1481,15.97,16.00,0


In [4]:
from src.training import train_model
from src.evaluation import evaluate_model

cv_config = deepcopy(base_config)

# Tidak menggunakan resampling pada eksperimen dasar ini.
cv_config["resampling"] = {"name": "none"}
cv_config["evaluation"]["threshold"] = 0.3

cv_records = []

for fold, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
    print(f"\nMemulai fold {fold}/5")

    # Mengambil bagian training dan evaluasi untuk fold ini.
    X_fold_train = X_cv.iloc[train_idx]
    y_fold_train = y_cv.iloc[train_idx]

    X_fold_valid = X_cv.iloc[valid_idx]
    y_fold_valid = y_cv.iloc[valid_idx]

    fold_config = deepcopy(cv_config)
    fold_config.update({
        "experiment_name": f"xgb_baseline_cv_fold_{fold}",
        "source_run_path": SOURCE_RUN_PATH,
        "train_sessions": len(train_idx),
        "validation_sessions": len(valid_idx),
        "cv": {
            "fold": fold,
            "n_splits": 5,
            "splitter": "StratifiedGroupKFold",
            "shuffle": False,
            "data_split": "original_train",
        },
    })

    with wandb.init(
        entity="elsaelisa",
        project="ecommerce-purchase-prediction",
        name=fold_config["experiment_name"],
        group="development_v2_xgb_baseline_cv",
        job_type="cross_validation",
        config=fold_config,
        mode="online",
    ) as run:

        # Membuat dan melatih pipeline baru untuk setiap fold.
        fold_pipeline = train_model(
            X_fold_train,
            y_fold_train,
            fold_config,
        )

        metrics, predictions, probabilities = evaluate_model(
            fold_pipeline,
            X_fold_valid,
            y_fold_valid,
            threshold=fold_config["evaluation"]["threshold"],
        )

        run.log({
            f"cv/{name}": float(value)
            for name, value in metrics.items()
        })

        cv_records.append({
            "fold": fold,
            "run_id": run.id,
            "run_url": run.url,
            **metrics,
        })

        print(f"F1: {metrics['f1']:.4f}")
        print(f"Average Precision: {metrics['average_precision']:.4f}")

cv_results = pd.DataFrame(cv_records)

display(
    cv_results.drop(columns=["run_id", "run_url"]).round(4)
)


Memulai fold 1/5


wandb: Currently logged in as: rifayel-126140185 (elsaelisa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


F1: 0.6742
Average Precision: 0.7157


cv/average_precision,▁
cv/f1,▁
cv/precision,▁
cv/recall,▁
cv/roc_auc,▁
cv/average_precision,0.71567
cv/f1,0.67416
cv/precision,0.60606
cv/recall,0.75949
cv/roc_auc,0.9229



Memulai fold 2/5


...

cv/average_precision,▁
cv/f1,▁
cv/precision,▁
cv/recall,▁
cv/roc_auc,▁
cv/average_precision,0.75282
cv/f1,0.67803
cv/precision,0.61512
cv/recall,0.75527
cv/roc_auc,0.93297



Memulai fold 5/5


F1: 0.7096
Average Precision: 0.7549


cv/average_precision,▁
cv/f1,▁
cv/precision,▁
cv/recall,▁
cv/roc_auc,▁
cv/average_precision,0.75487
cv/f1,0.70955
cv/precision,0.65942
cv/recall,0.76793
cv/roc_auc,0.93299


,fold,precision,recall,f1,average_precision,roc_auc
0,1,0.6061,0.7595,0.6742,0.7157,0.9229
1,2,0.6107,0.7246,0.6628,0.7177,0.9244
2,3,0.6667,0.7288,0.6964,0.7759,0.9378
3,4,0.6151,0.7553,0.6780,0.7528,0.9330
4,5,0.6594,0.7679,0.7096,0.7549,0.9330


In [5]:
metric_columns = [
    "precision",
    "recall",
    "f1",
    "average_precision",
    "roc_auc",
]

cv_summary = (
    cv_results[metric_columns]
    .agg(["mean", "std", "min", "max"])
    .T
)

cv_summary.index.name = "metric"

display(cv_summary.round(4))

,mean,std,min,max
metric,,,,
precision,0.6316,0.0290,0.6061,0.6667
recall,0.7472,0.0193,0.7246,0.7679
f1,0.6842,0.0186,0.6628,0.7096
average_precision,0.7434,0.0260,0.7157,0.7759
roc_auc,0.9302,0.0063,0.9229,0.9378


In [6]:
with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    name="xgb_baseline_cv_summary",
    group="development_v2_xgb_baseline_cv",
    job_type="cv_summary",
    mode="online",
    config={
        "source_run_path": SOURCE_RUN_PATH,
        "source_fold_run_ids": cv_results["run_id"].tolist(),
        "training_config": cv_config,
        "n_splits": 5,
        "splitter": "StratifiedGroupKFold",
        "shuffle": False,
        "data_split": "original_train",
        "threshold": 0.3,
        "purpose": "post_selection_development_diagnostic",
    },
) as run:

    run.log({
        "cv/fold_results": wandb.Table(dataframe=cv_results),
        "cv/summary": wandb.Table(
            dataframe=cv_summary.reset_index()
        ),
    })

    for metric in metric_columns:
        run.summary[f"cv_mean/{metric}"] = float(
            cv_summary.loc[metric, "mean"]
        )
        run.summary[f"cv_std/{metric}"] = float(
            cv_summary.loc[metric, "std"]
        )

    # Menyimpan laporan lokal dalam folder run ini.
    report_dir = PROJECT_ROOT / "outputs" / run.id
    report_dir.mkdir(parents=True, exist_ok=True)

    results_path = report_dir / "cv_fold_results.csv"
    summary_path = report_dir / "cv_summary.csv"

    cv_results.to_csv(results_path, index=False)
    cv_summary.to_csv(summary_path)

    artifact = wandb.Artifact(
        name="xgb-baseline-cv-report",
        type="evaluation",
    )
    artifact.add_file(str(results_path))
    artifact.add_file(str(summary_path))
    run.log_artifact(artifact)

    print("Laporan disimpan:", report_dir)
    print("Link W&B:", run.url)

Laporan disimpan: C:\Users\elsae\Downloads\ecommerce-purchase-prediction\outputs\ilfnxcdk
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/ilfnxcdk


cv_mean/average_precision,0.74338
cv_mean/f1,0.68418
cv_mean/precision,0.6316
cv_mean/recall,0.74722
cv_mean/roc_auc,0.9302
cv_std/average_precision,0.02601
cv_std/f1,0.01863
cv_std/precision,0.029
cv_std/recall,0.01934
cv_std/roc_auc,0.00633


In [7]:
from sklearn.model_selection import ParameterGrid

parameter_grid = {
    "n_estimators": [200, 300],
    "max_depth": [3, 4],
    "learning_rate": [0.03, 0.05],
}

search_candidates = list(ParameterGrid(parameter_grid))

candidate_table = pd.DataFrame(search_candidates)

candidate_table.insert(
    0,
    "candidate_id",
    range(1, len(search_candidates) + 1),
)

# Menandai kombinasi yang sudah dievaluasi pada CV sebelumnya.
baseline_params = cv_config["model"]["params"]

candidate_table["is_baseline"] = [
    all(
        candidate[param] == baseline_params[param]
        for param in parameter_grid
    )
    for candidate in search_candidates
]

display(candidate_table)

new_candidates = int((~candidate_table["is_baseline"]).sum())

print("Jumlah kombinasi:", len(search_candidates))
print("Kombinasi baru:", new_candidates)
print("Training baru yang diperlukan:", new_candidates * len(cv_splits))

,candidate_id,learning_rate,max_depth,n_estimators,is_baseline
0,1,0.03,3,200,False
1,2,0.03,3,300,False
2,3,0.03,4,200,False
3,4,0.03,4,300,False
4,5,0.05,3,200,False
5,6,0.05,3,300,False
6,7,0.05,4,200,False
7,8,0.05,4,300,True


Jumlah kombinasi: 8
Kombinasi baru: 7
Training baru yang diperlukan: 35


In [8]:
import importlib
import src.training as training_module

# Membaca perubahan terbaru pada src/training.py.
importlib.reload(training_module)


def run_tuning_candidate(candidate_id):
    params = search_candidates[candidate_id - 1]

    config = deepcopy(cv_config)
    config["model"]["params"].update(params)
    config["experiment_name"] = (
        f"xgb_tuning_candidate_{candidate_id:02d}"
    )
    config["cv"] = {
        "n_splits": 5,
        "splitter": "StratifiedGroupKFold",
        "shuffle": False,
        "data_split": "original_train",
    }

    with wandb.init(
        entity="elsaelisa",
        project="ecommerce-purchase-prediction",
        name=config["experiment_name"],
        group="development_v2_xgb_tuning",
        job_type="hyperparameter_tuning",
        config=config,
        mode="online",
    ) as run:

        def log_fold(result):
            run.log({
                f"cv_fold/{key}": float(value)
                for key, value in result.items()
            })

        fold_results = training_module.cross_validate_model(
            X=X_cv,
            y=y_cv,
            cv_splits=cv_splits,
            config=config,
            on_fold_end=log_fold,
        )

        fold_df = pd.DataFrame(fold_results)

        record = {
            "candidate_id": candidate_id,
            **params,
            "run_id": run.id,
            "run_url": run.url,
        }

        for metric in metric_columns:
            mean_value = float(fold_df[metric].mean())
            std_value = float(fold_df[metric].std())

            record[f"mean_{metric}"] = mean_value
            record[f"std_{metric}"] = std_value

            run.summary[f"cv_mean/{metric}"] = mean_value
            run.summary[f"cv_std/{metric}"] = std_value

        run.log({
            "cv/fold_results": wandb.Table(dataframe=fold_df)
        })

        report_dir = PROJECT_ROOT / "outputs" / run.id
        report_dir.mkdir(parents=True, exist_ok=True)

        report_path = report_dir / "cv_fold_results.csv"
        fold_df.to_csv(report_path, index=False)

        artifact = wandb.Artifact(
            name=f"xgb-tuning-candidate-{candidate_id:02d}",
            type="evaluation",
        )
        artifact.add_file(str(report_path))
        run.log_artifact(artifact)

        print("Link W&B:", run.url)

    return record

In [9]:
tuning_records = []

candidate_result = run_tuning_candidate(candidate_id=1)

tuning_records.append(candidate_result)

display(
    pd.DataFrame(tuning_records)[[
        "candidate_id",
        "learning_rate",
        "max_depth",
        "n_estimators",
        "mean_f1",
        "std_f1",
        "mean_average_precision",
    ]].round(4)
)

Fold 1/5 | F1: 0.6814
Fold 2/5 | F1: 0.6679
Fold 3/5 | F1: 0.7107
Fold 4/5 | F1: 0.6987
Fold 5/5 | F1: 0.7077
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/eesnl0o2


cv_fold/average_precision,▁▂█▆▆
cv_fold/f1,▃▁█▆█
cv_fold/fold,▁▃▅▆█
cv_fold/precision,▁▁█▃▇
cv_fold/recall,▆▁▅█▅
cv_fold/roc_auc,▁▂█▆▆
cv_fold/average_precision,0.76619
cv_fold/f1,0.70769
cv_fold/fold,5
cv_fold/precision,0.65018
cv_fold/recall,0.77637


,candidate_id,learning_rate,max_depth,n_estimators,mean_f1,std_f1,mean_average_precision
0,1,0.03,3,200,0.6933,0.0182,0.7498


In [10]:
# Melewati kandidat yang sudah selesai dan kandidat baseline.
completed_ids = {
    int(record["candidate_id"])
    for record in tuning_records
}

pending_ids = candidate_table.loc[
    ~candidate_table["is_baseline"]
    & ~candidate_table["candidate_id"].isin(completed_ids),
    "candidate_id",
].tolist()

# Menyimpan progres setiap kali satu kandidat selesai.
progress_dir = PROJECT_ROOT / "outputs" / "reports"
progress_dir.mkdir(parents=True, exist_ok=True)

progress_path = progress_dir / "xgb_tuning_progress.csv"

for candidate_id in pending_ids:
    print(f"\n=== Kandidat {candidate_id} ===")

    result = run_tuning_candidate(
        candidate_id=int(candidate_id)
    )

    tuning_records.append(result)

    pd.DataFrame(tuning_records).to_csv(
        progress_path,
        index=False,
    )

tuning_results = pd.DataFrame(tuning_records)

display(
    tuning_results[[
        "candidate_id",
        "learning_rate",
        "max_depth",
        "n_estimators",
        "mean_f1",
        "std_f1",
        "mean_average_precision",
    ]]
    .sort_values(
        ["mean_f1", "mean_average_precision"],
        ascending=False,
    )
    .round(4)
)

print("Progres disimpan:", progress_path)


=== Kandidat 2 ===


Fold 1/5 | F1: 0.6939
Fold 2/5 | F1: 0.6705
Fold 3/5 | F1: 0.7098
Fold 4/5 | F1: 0.6974
Fold 5/5 | F1: 0.7146
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/zf9fydza


cv_fold/average_precision,▁▂█▆▆
cv_fold/f1,▅▁▇▅█
cv_fold/fold,▁▃▅▆█
cv_fold/precision,▃▁█▃█
cv_fold/recall,▇▁▃█▅
cv_fold/roc_auc,▁▃█▆▆
cv_fold/average_precision,0.76558
cv_fold/f1,0.71456
cv_fold/fold,5
cv_fold/precision,0.66187
cv_fold/recall,0.77637



=== Kandidat 3 ===


Fold 1/5 | F1: 0.6729
Fold 2/5 | F1: 0.6818
Fold 3/5 | F1: 0.7137
Fold 4/5 | F1: 0.6929
Fold 5/5 | F1: 0.7107
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/ydy6xfyo


...


=== Kandidat 7 ===


Fold 1/5 | F1: 0.6803
Fold 2/5 | F1: 0.6756
Fold 3/5 | F1: 0.7092
Fold 4/5 | F1: 0.6890
Fold 5/5 | F1: 0.7079
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/t3n4zyst


cv_fold/average_precision,▁▂█▅▆
cv_fold/f1,▂▁█▄█
cv_fold/fold,▁▃▅▆█
cv_fold/precision,▁▂█▂▆
cv_fold/recall,▆▁▃█▆
cv_fold/roc_auc,▁▂█▆▅
cv_fold/average_precision,0.76022
cv_fold/f1,0.70793
cv_fold/fold,5
cv_fold/precision,0.65357
cv_fold/recall,0.77215


,candidate_id,learning_rate,max_depth,n_estimators,mean_f1,std_f1,mean_average_precision
1,2,0.03,3,300,0.6972,0.0172,0.7476
3,4,0.03,4,300,0.6951,0.0196,0.7509
2,3,0.03,4,200,0.6944,0.0178,0.7529
5,6,0.05,3,300,0.6943,0.0150,0.7456
0,1,0.03,3,200,0.6933,0.0182,0.7498
6,7,0.05,4,200,0.6924,0.0155,0.7493
4,5,0.05,3,200,0.6913,0.0136,0.7493


Progres disimpan: C:\Users\elsae\Downloads\ecommerce-purchase-prediction\outputs\reports\xgb_tuning_progress.csv


In [11]:
# Mengambil nomor kandidat baseline dari tabel pengaturan.
baseline_id = int(
    candidate_table.loc[
        candidate_table["is_baseline"],
        "candidate_id",
    ].iloc[0]
)

baseline_record = {
    "candidate_id": baseline_id,
    **search_candidates[baseline_id - 1],
}

# Menggunakan kembali hasil CV baseline yang sudah dihitung.
for metric in metric_columns:
    baseline_record[f"mean_{metric}"] = float(
        cv_summary.loc[metric, "mean"]
    )
    baseline_record[f"std_{metric}"] = float(
        cv_summary.loc[metric, "std"]
    )

# Menggabungkan tujuh kandidat baru dengan satu baseline.
comparison_results = pd.concat(
    [
        tuning_results.copy(),
        pd.DataFrame([baseline_record]),
    ],
    ignore_index=True,
)

comparison_results = comparison_results.sort_values(
    ["mean_f1", "mean_average_precision", "candidate_id"],
    ascending=[False, False, True],
).reset_index(drop=True)

best_candidate_id = int(
    comparison_results.iloc[0]["candidate_id"]
)

display(
    comparison_results[[
        "candidate_id",
        "learning_rate",
        "max_depth",
        "n_estimators",
        "mean_f1",
        "std_f1",
        "mean_average_precision",
    ]].round(4)
)

print("Kandidat terpilih:", best_candidate_id)

,candidate_id,learning_rate,max_depth,n_estimators,mean_f1,std_f1,mean_average_precision
0,2,0.03,3,300,0.6972,0.0172,0.7476
1,4,0.03,4,300,0.6951,0.0196,0.7509
2,3,0.03,4,200,0.6944,0.0178,0.7529
3,6,0.05,3,300,0.6943,0.0150,0.7456
4,1,0.03,3,200,0.6933,0.0182,0.7498
5,7,0.05,4,200,0.6924,0.0155,0.7493
6,5,0.05,3,200,0.6913,0.0136,0.7493
7,8,0.05,4,300,0.6842,0.0186,0.7434


Kandidat terpilih: 2


In [12]:
import yaml

# Menyiapkan konfigurasi XGBoost hasil tuning.
tuned_config = deepcopy(cv_config)

tuned_config["experiment_name"] = "xgb_tuned_all_features"

tuned_config["model"]["params"].update(
    search_candidates[best_candidate_id - 1]
)

tuned_config["selection"] = {
    "candidate_id": best_candidate_id,
    "metric": "mean_f1",
    "method": "5_fold_group_cross_validation",
    "data_split": "original_train",
}

config_path = (
    PROJECT_ROOT / "configs" / "xgb_tuned_all_features.yaml"
)

with config_path.open("w", encoding="utf-8") as file:
    yaml.safe_dump(tuned_config, file, sort_keys=False)

# Mencatat perbandingan lengkap ke W&B.
with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    name="xgb_tuning_all_features_summary",
    group="development_v2_xgb_tuning",
    job_type="tuning_summary",
    mode="online",
    config={
        "selected_config": tuned_config,
        "baseline_fold_run_ids": cv_results["run_id"].tolist(),
        "tuning_run_ids": tuning_results["run_id"].tolist(),
        "parameter_grid": parameter_grid,
    },
) as run:

    run.log({
        "tuning/comparison": wandb.Table(
            dataframe=comparison_results
        )
    })

    run.summary["selected/candidate_id"] = best_candidate_id
    run.summary["selected/cv_mean_f1"] = float(
        comparison_results.iloc[0]["mean_f1"]
    )
    run.summary["selected/cv_std_f1"] = float(
        comparison_results.iloc[0]["std_f1"]
    )

    report_dir = PROJECT_ROOT / "outputs" / run.id
    report_dir.mkdir(parents=True, exist_ok=True)

    comparison_path = report_dir / "tuning_comparison.csv"

    comparison_results.to_csv(
        comparison_path,
        index=False,
    )

    artifact = wandb.Artifact(
        name="xgb-tuning-all-features-report",
        type="evaluation",
    )
    artifact.add_file(str(comparison_path))
    artifact.add_file(str(config_path))

    run.log_artifact(artifact)

    print("Konfigurasi disimpan:", config_path)
    print("Link W&B:", run.url)

Konfigurasi disimpan: C:\Users\elsae\Downloads\ecommerce-purchase-prediction\configs\xgb_tuned_all_features.yaml
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/19cqn0pg


selected/candidate_id,2
selected/cv_mean_f1,0.69722
selected/cv_std_f1,0.01722


In [13]:
# Menghapus PageValues dari fitur training.
X_without_pagevalues = X_cv.drop(columns=["PageValues"])

# Menyalin pengaturan model hasil tuning.
without_pagevalues_config = deepcopy(tuned_config)

without_pagevalues_config["experiment_name"] = (
    "xgb_tuned_without_pagevalues_cv"
)

without_pagevalues_config["features"]["exclude_columns"] = (
    list(dict.fromkeys(
        tuned_config["features"]["exclude_columns"]
        + ["PageValues"]
    ))
)

# Memperbarui daftar fitur yang dicatat.
without_pagevalues_config["feature_columns"] = (
    X_without_pagevalues.columns.tolist()
)

# Menjelaskan bahwa parameter berasal dari tuning seluruh fitur.
without_pagevalues_config["selection"] = {
    "purpose": "controlled_feature_ablation",
    "parameter_source": "configs/xgb_tuned_all_features.yaml",
    "source_candidate_id": best_candidate_id,
}

assert X_without_pagevalues.index.equals(X_cv.index)

print("Dengan PageValues:", X_cv.shape)
print("Tanpa PageValues:", X_without_pagevalues.shape)
print(
    "Threshold:",
    without_pagevalues_config["evaluation"]["threshold"],
)
print(
    "PageValues masih tersedia:",
    "PageValues" in X_without_pagevalues.columns,
)

Dengan PageValues: (7406, 17)
Tanpa PageValues: (7406, 16)
Threshold: 0.3
PageValues masih tersedia: False


In [14]:
without_pagevalues_config["cv"] = {
    "n_splits": 5,
    "splitter": "StratifiedGroupKFold",
    "shuffle": False,
    "data_split": "original_train",
}

with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    name=without_pagevalues_config["experiment_name"],
    group="development_v2_feature_ablation",
    job_type="cross_validation",
    config=without_pagevalues_config,
    mode="online",
) as run:

    def log_without_pagevalues_fold(result):
        run.log({
            f"cv_fold/{key}": float(value)
            for key, value in result.items()
        })

    fold_results = training_module.cross_validate_model(
        X=X_without_pagevalues,
        y=y_cv,
        cv_splits=cv_splits,
        config=without_pagevalues_config,
        on_fold_end=log_without_pagevalues_fold,
    )

    without_pagevalues_results = pd.DataFrame(fold_results)

    without_pagevalues_summary = (
        without_pagevalues_results[metric_columns]
        .agg(["mean", "std", "min", "max"])
        .T
    )
    without_pagevalues_summary.index.name = "metric"

    run.log({
        "cv/fold_results": wandb.Table(
            dataframe=without_pagevalues_results
        ),
        "cv/summary": wandb.Table(
            dataframe=without_pagevalues_summary.reset_index()
        ),
    })

    for metric in metric_columns:
        for statistic in ["mean", "std"]:
            run.summary[f"cv_{statistic}/{metric}"] = float(
                without_pagevalues_summary.loc[metric, statistic]
            )

    report_dir = PROJECT_ROOT / "outputs" / run.id
    report_dir.mkdir(parents=True, exist_ok=True)

    results_path = report_dir / "cv_fold_results.csv"
    summary_path = report_dir / "cv_summary.csv"

    without_pagevalues_results.to_csv(results_path, index=False)
    without_pagevalues_summary.to_csv(summary_path)

    artifact = wandb.Artifact(
        name="xgb-without-pagevalues-cv-report",
        type="evaluation",
    )
    artifact.add_file(str(results_path))
    artifact.add_file(str(summary_path))
    run.log_artifact(artifact)

    without_pagevalues_run_id = run.id

    print("Link W&B:", run.url)

display(without_pagevalues_summary.round(4))

Fold 1/5 | F1: 0.3864
Fold 2/5 | F1: 0.4200
Fold 3/5 | F1: 0.4237
Fold 4/5 | F1: 0.4053
Fold 5/5 | F1: 0.3888
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/ml3zsobu


cv_fold/average_precision,▅▁▆█▅
cv_fold/f1,▁▇█▅▁
cv_fold/fold,▁▃▅▆█
cv_fold/precision,▁▆██▃
cv_fold/recall,▂█▇▂▁
cv_fold/roc_auc,▃▂█▅▁
cv_fold/average_precision,0.38493
cv_fold/f1,0.38877
cv_fold/fold,5
cv_fold/precision,0.39823
cv_fold/recall,0.37975


,mean,std,min,max
metric,,,,
precision,0.4094,0.0156,0.3889,0.4240
recall,0.4007,0.0232,0.3797,0.4280
f1,0.4048,0.0172,0.3864,0.4237
average_precision,0.3840,0.0215,0.3497,0.4082
roc_auc,0.7838,0.0135,0.7709,0.8044


In [15]:
# Menyimpan hasil kandidat 2 tanpa PageValues yang sudah selesai.
existing_result = {
    "candidate_id": best_candidate_id,
    **search_candidates[best_candidate_id - 1],
    "run_id": without_pagevalues_run_id,
    "run_url": (
        "https://wandb.ai/elsaelisa/"
        "ecommerce-purchase-prediction/runs/"
        + without_pagevalues_run_id
    ),
}

for metric in metric_columns:
    for statistic in ["mean", "std"]:
        existing_result[f"{statistic}_{metric}"] = float(
            without_pagevalues_summary.loc[metric, statistic]
        )

without_pagevalues_tuning_records = [existing_result]

# Semua kandidat selain kandidat yang sudah dievaluasi.
pending_without_pagevalues_ids = [
    candidate_id
    for candidate_id in range(1, len(search_candidates) + 1)
    if candidate_id != best_candidate_id
]

print("Kandidat yang sudah selesai:", best_candidate_id)
print("Kandidat berikutnya:", pending_without_pagevalues_ids)
print(
    "Jumlah training tambahan:",
    len(pending_without_pagevalues_ids) * len(cv_splits),
)

Kandidat yang sudah selesai: 2
Kandidat berikutnya: [1, 3, 4, 5, 6, 7, 8]
Jumlah training tambahan: 35


In [16]:
def run_feature_tuning_candidate(
    candidate_id,
    X_input,
    config_template,
    feature_variant,
):
    params = search_candidates[candidate_id - 1]

    config = deepcopy(config_template)
    config["model"]["params"].update(params)
    config["feature_columns"] = X_input.columns.tolist()
    config["experiment_name"] = (
        f"xgb_{feature_variant}_candidate_{candidate_id:02d}"
    )
    config["selection"] = {
        "purpose": "hyperparameter_tuning",
        "metric": "mean_f1",
        "data_split": "original_train",
    }

    with wandb.init(
        entity="elsaelisa",
        project="ecommerce-purchase-prediction",
        name=config["experiment_name"],
        group=f"development_v2_tuning_{feature_variant}",
        job_type="hyperparameter_tuning",
        config=config,
        mode="online",
    ) as run:

        def log_fold(result):
            run.log({
                f"cv_fold/{key}": float(value)
                for key, value in result.items()
            })

        fold_results = training_module.cross_validate_model(
            X=X_input,
            y=y_cv,
            cv_splits=cv_splits,
            config=config,
            on_fold_end=log_fold,
        )

        fold_df = pd.DataFrame(fold_results)

        record = {
            "candidate_id": candidate_id,
            **params,
            "run_id": run.id,
            "run_url": run.url,
        }

        for metric in metric_columns:
            for statistic in ["mean", "std"]:
                value = float(
                    fold_df[metric].agg(statistic)
                )
                record[f"{statistic}_{metric}"] = value
                run.summary[f"cv_{statistic}/{metric}"] = value

        run.log({
            "cv/fold_results": wandb.Table(dataframe=fold_df)
        })

        report_dir = PROJECT_ROOT / "outputs" / run.id
        report_dir.mkdir(parents=True, exist_ok=True)

        report_path = report_dir / "cv_fold_results.csv"
        fold_df.to_csv(report_path, index=False)

        artifact = wandb.Artifact(
            name=f"xgb-{feature_variant}-candidate-{candidate_id:02d}",
            type="evaluation",
        )
        artifact.add_file(str(report_path))
        run.log_artifact(artifact)

        print("Link W&B:", run.url)

    return record

In [17]:
completed_ids = {
    int(record["candidate_id"])
    for record in without_pagevalues_tuning_records
}

progress_path_without_pagevalues = (
    progress_dir / "xgb_without_pagevalues_tuning_progress.csv"
)

for candidate_id in pending_without_pagevalues_ids:
    if candidate_id in completed_ids:
        continue

    print(f"\n=== Tanpa PageValues: kandidat {candidate_id} ===")

    result = run_feature_tuning_candidate(
        candidate_id=candidate_id,
        X_input=X_without_pagevalues,
        config_template=without_pagevalues_config,
        feature_variant="without_pagevalues",
    )

    without_pagevalues_tuning_records.append(result)
    completed_ids.add(candidate_id)

    pd.DataFrame(without_pagevalues_tuning_records).to_csv(
        progress_path_without_pagevalues,
        index=False,
    )

without_pagevalues_tuning_results = pd.DataFrame(
    without_pagevalues_tuning_records
)

display(
    without_pagevalues_tuning_results[[
        "candidate_id",
        "learning_rate",
        "max_depth",
        "n_estimators",
        "mean_f1",
        "std_f1",
        "mean_average_precision",
    ]]
    .sort_values(
        ["mean_f1", "mean_average_precision", "candidate_id"],
        ascending=[False, False, True],
    )
    .round(4)
)


=== Tanpa PageValues: kandidat 1 ===


Fold 1/5 | F1: 0.3608
Fold 2/5 | F1: 0.4051
Fold 3/5 | F1: 0.4139
Fold 4/5 | F1: 0.3918
Fold 5/5 | F1: 0.3803
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/nl0in760


cv_fold/average_precision,▅▁▆█▆
cv_fold/f1,▁▇█▅▄
cv_fold/fold,▁▃▅▆█
cv_fold/precision,▁▅██▅
cv_fold/recall,▁██▃▃
cv_fold/roc_auc,▂▂█▅▁
cv_fold/average_precision,0.38625
cv_fold/f1,0.38031
cv_fold/fold,5
cv_fold/precision,0.40476
cv_fold/recall,0.35865



=== Tanpa PageValues: kandidat 3 ===


Fold 1/5 | F1: 0.3941
Fold 2/5 | F1: 0.4292
Fold 3/5 | F1: 0.4101
Fold 4/5 | F1: 0.3929
Fold 5/5 | F1: 0.4059
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/f94i9dv4


...

cv_fold/average_precision,▅▁▇█▄
cv_fold/f1,▁▃█▄▂
cv_fold/fold,▁▃▅▆█
cv_fold/precision,▁▂▅█▃
cv_fold/recall,▃▅█▁▃
cv_fold/roc_auc,▄▂█▅▁
cv_fold/average_precision,0.37852
cv_fold/f1,0.39419
cv_fold/fold,5
cv_fold/precision,0.38776
cv_fold/recall,0.40084



=== Tanpa PageValues: kandidat 8 ===


Fold 1/5 | F1: 0.4024
Fold 2/5 | F1: 0.3943
Fold 3/5 | F1: 0.4315
Fold 4/5 | F1: 0.3846
Fold 5/5 | F1: 0.3968
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/uhle0lt9


cv_fold/average_precision,▄▁▇█▂
cv_fold/f1,▄▂█▁▃
cv_fold/fold,▁▃▅▆█
cv_fold/precision,▃▁█▃▁
cv_fold/recall,▅▄█▁▄
cv_fold/roc_auc,▄▁█▆▁
cv_fold/average_precision,0.366
cv_fold/f1,0.39676
cv_fold/fold,5
cv_fold/precision,0.38132
cv_fold/recall,0.4135


,candidate_id,learning_rate,max_depth,n_estimators,mean_f1,std_f1,mean_average_precision
2,3,0.03,4,200,0.4065,0.0147,0.3832
0,2,0.03,3,300,0.4048,0.0172,0.3840
5,6,0.05,3,300,0.4031,0.0132,0.3836
3,4,0.03,4,300,0.4025,0.0090,0.3840
7,8,0.05,4,300,0.4019,0.0177,0.3858
6,7,0.05,4,200,0.3987,0.0081,0.3873
4,5,0.05,3,200,0.3980,0.0160,0.3815
1,1,0.03,3,200,0.3904,0.0209,0.3815


In [18]:
# Mengurutkan kandidat sesuai kriteria yang sudah ditetapkan.
without_pagevalues_comparison = (
    without_pagevalues_tuning_results
    .sort_values(
        ["mean_f1", "mean_average_precision", "candidate_id"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

best_without_pagevalues_id = int(
    without_pagevalues_comparison.iloc[0]["candidate_id"]
)

# Menyiapkan konfigurasi terbaik tanpa PageValues.
tuned_without_pagevalues_config = deepcopy(
    without_pagevalues_config
)

tuned_without_pagevalues_config["experiment_name"] = (
    "xgb_tuned_without_pagevalues"
)

tuned_without_pagevalues_config["model"]["params"].update(
    search_candidates[best_without_pagevalues_id - 1]
)

tuned_without_pagevalues_config["selection"] = {
    "candidate_id": best_without_pagevalues_id,
    "metric": "mean_f1",
    "method": "5_fold_group_cross_validation",
    "data_split": "original_train",
}

config_path_without_pagevalues = (
    PROJECT_ROOT
    / "configs"
    / "xgb_tuned_without_pagevalues.yaml"
)

with config_path_without_pagevalues.open(
    "w", encoding="utf-8"
) as file:
    yaml.safe_dump(
        tuned_without_pagevalues_config,
        file,
        sort_keys=False,
    )

with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    name="xgb_tuning_without_pagevalues_summary",
    group="development_v2_tuning_without_pagevalues",
    job_type="tuning_summary",
    mode="online",
    config={
        "selected_config": tuned_without_pagevalues_config,
        "source_run_ids": (
            without_pagevalues_comparison["run_id"].tolist()
        ),
        "parameter_grid": parameter_grid,
    },
) as run:

    run.log({
        "tuning/comparison": wandb.Table(
            dataframe=without_pagevalues_comparison
        )
    })

    run.summary["selected/candidate_id"] = (
        best_without_pagevalues_id
    )
    run.summary["selected/cv_mean_f1"] = float(
        without_pagevalues_comparison.iloc[0]["mean_f1"]
    )

    report_dir = PROJECT_ROOT / "outputs" / run.id
    report_dir.mkdir(parents=True, exist_ok=True)

    comparison_path = report_dir / "tuning_comparison.csv"

    without_pagevalues_comparison.to_csv(
        comparison_path,
        index=False,
    )

    artifact = wandb.Artifact(
        name="xgb-tuning-without-pagevalues-report",
        type="evaluation",
    )
    artifact.add_file(str(comparison_path))
    artifact.add_file(str(config_path_without_pagevalues))
    run.log_artifact(artifact)

    print("Kandidat terpilih:", best_without_pagevalues_id)
    print("Konfigurasi:", config_path_without_pagevalues)
    print("Link W&B:", run.url)

Kandidat terpilih: 3
Konfigurasi: C:\Users\elsae\Downloads\ecommerce-purchase-prediction\configs\xgb_tuned_without_pagevalues.yaml
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/ebyc4hf5


selected/candidate_id,3
selected/cv_mean_f1,0.40645


In [19]:
import shutil

active_config_path = (
    PROJECT_ROOT / "configs" / "experiment.yaml"
)

backup_config_path = (
    PROJECT_ROOT / "configs" / "experiment_before_v2.yaml"
)

# Menyimpan konfigurasi sebelumnya satu kali.
if active_config_path.exists() and not backup_config_path.exists():
    shutil.copy2(active_config_path, backup_config_path)

# Mengaktifkan konfigurasi hasil tuning seluruh fitur.
shutil.copy2(
    PROJECT_ROOT / "configs" / "xgb_tuned_all_features.yaml",
    active_config_path,
)

with active_config_path.open("r", encoding="utf-8") as file:
    active_config = yaml.safe_load(file)

print("Eksperimen:", active_config["experiment_name"])
print("Parameter:", active_config["model"]["params"])
print("Threshold:", active_config["evaluation"]["threshold"])

Eksperimen: xgb_tuned_all_features
Parameter: {'device': 'cpu', 'n_jobs': -1, 'max_depth': 3, 'objective': 'binary:logistic', 'subsample': 0.8, 'reg_lambda': 1, 'eval_metric': 'logloss', 'tree_method': 'hist', 'n_estimators': 300, 'learning_rate': 0.03, 'colsample_bytree': 0.8, 'min_child_weight': 1, 'scale_pos_weight': 1}
Threshold: 0.3


In [20]:
# Mengaktifkan konfigurasi hasil tuning tanpa PageValues.
shutil.copy2(
    PROJECT_ROOT / "configs" / "xgb_tuned_without_pagevalues.yaml",
    active_config_path,
)

with active_config_path.open("r", encoding="utf-8") as file:
    active_config = yaml.safe_load(file)

print("Eksperimen:", active_config["experiment_name"])
print(
    "Fitur yang dihapus:",
    active_config["features"]["exclude_columns"],
)
print("Threshold:", active_config["evaluation"]["threshold"])

Eksperimen: xgb_tuned_without_pagevalues
Fitur yang dihapus: ['PageValues']
Threshold: 0.3


In [21]:
# Menggabungkan hasil CV dan validation kedua varian.
run_specs = [
    (
        "all_features",
        "22cse45y",
        comparison_results.iloc[0],
    ),
    (
        "without_pagevalues",
        "zruv9kiq",
        without_pagevalues_comparison.iloc[0],
    ),
]

development_records = []

for variant, run_id, cv_row in run_specs:
    run_path = (
        f"elsaelisa/ecommerce-purchase-prediction/{run_id}"
    )
    saved_run = wandb.Api().run(run_path)

    record = {
        "variant": variant,
        "source_run_path": run_path,
        "threshold": float(
            saved_run.config["evaluation"]["threshold"]
        ),
        "cv_mean_f1": float(cv_row["mean_f1"]),
        "cv_std_f1": float(cv_row["std_f1"]),
    }

    for metric in metric_columns:
        record[f"validation_{metric}"] = float(
            saved_run.summary[f"validation/{metric}"]
        )

    development_records.append(record)

development_comparison = pd.DataFrame(development_records)

display(development_comparison.round(4))

,variant,source_run_path,threshold,cv_mean_f1,cv_std_f1,validation_precision,validation_recall,validation_f1,validation_average_precision,validation_roc_auc
0,all_features,elsaelisa/ecommerce-purchase-prediction/22cse45y,0.3,0.6972,0.0172,0.5726,0.7676,0.6559,0.7419,0.9331
1,without_pagevalues,elsaelisa/ecommerce-purchase-prediction/zruv9kiq,0.3,0.4065,0.0147,0.3392,0.3622,0.3503,0.3299,0.7714


In [22]:
with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    name="development_v2_comparison",
    job_type="analysis",
    mode="online",
    config={
        "source_run_paths": (
            development_comparison["source_run_path"].tolist()
        ),
        "cv_data_split": "original_train",
        "evaluation_split": "original_validation",
        "threshold": 0.3,
    },
) as run:

    run.log({
        "comparison/results": wandb.Table(
            dataframe=development_comparison
        )
    })

    report_dir = PROJECT_ROOT / "outputs" / run.id
    report_dir.mkdir(parents=True, exist_ok=True)

    report_path = report_dir / "development_v2_comparison.csv"

    development_comparison.to_csv(report_path, index=False)

    artifact = wandb.Artifact(
        name="development-v2-comparison",
        type="evaluation",
    )
    artifact.add_file(str(report_path))
    run.log_artifact(artifact)

    print("Laporan:", report_path)
    print("Link W&B:", run.url)

Laporan: C:\Users\elsae\Downloads\ecommerce-purchase-prediction\outputs\xkcyl4jj\development_v2_comparison.csv
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/xkcyl4jj


In [23]:
import importlib
import src.models as models_module
import src.training as training_module

# Memuat perubahan registry dan fungsi training.
importlib.reload(models_module)
importlib.reload(training_module)

catboost_config = {
    "experiment_name": "catboost_all_features",
    "seed": int(cv_config["seed"]),
    "data": deepcopy(cv_config["data"]),
    "features": {
        "exclude_columns": [],
    },
    "model": {
        "name": "catboost",
        "params": {
            "iterations": 300,
            "depth": 4,
            "learning_rate": 0.05,
            "l2_leaf_reg": 3.0,
            "loss_function": "Logloss",
            "task_type": "CPU",
            "thread_count": 2,
            "verbose": False,
            "allow_writing_files": False,
        },
    },
    "resampling": {
        "name": "none",
    },
    "evaluation": {
        "threshold": 0.3,
    },
    "wandb": {
        "project": "ecommerce-purchase-prediction",
        "mode": "online",
    },
    "output_dir": "outputs",
}

catboost_config_path = (
    PROJECT_ROOT / "configs" / "catboost_all_features.yaml"
)

with catboost_config_path.open("w", encoding="utf-8") as file:
    yaml.safe_dump(catboost_config, file, sort_keys=False)

print("Konfigurasi disimpan:", catboost_config_path)

Konfigurasi disimpan: C:\Users\elsae\Downloads\ecommerce-purchase-prediction\configs\catboost_all_features.yaml


In [24]:
preview_pipeline = training_module.build_catboost_pipeline(
    X_cv,
    catboost_config,
)

X_catboost_preview = (
    preview_pipeline.named_steps["preprocessor"]
    .fit_transform(X_cv)
)

print(
    "Model:",
    type(preview_pipeline.named_steps["model"]).__name__,
)
print("Ukuran sebelum preprocessing:", X_cv.shape)
print("Ukuran setelah preprocessing:", X_catboost_preview.shape)
print("Tipe hasil:", type(X_catboost_preview).__name__)

display(
    X_catboost_preview[
        training_module.CATEGORICAL_COLUMNS
    ].dtypes
)

Model: CatBoostClassifier
Ukuran sebelum preprocessing: (7406, 17)
Ukuran setelah preprocessing: (7406, 17)
Tipe hasil: DataFrame


Month               object
OperatingSystems    object
Browser             object
Region              object
TrafficType         object
VisitorType         object
Weekend             object
dtype: object

In [25]:
from importlib.metadata import version

catboost_cv_config = deepcopy(catboost_config)
catboost_cv_config["experiment_name"] = "catboost_all_features_cv"

catboost_cv_config["feature_columns"] = X_cv.columns.tolist()

catboost_cv_config["cv"] = {
    "n_splits": 5,
    "splitter": "StratifiedGroupKFold",
    "shuffle": False,
    "data_split": "original_train",
}

catboost_cv_config["library_versions"] = {
    "catboost": version("catboost"),
    "scikit-learn": version("scikit-learn"),
}

with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    name=catboost_cv_config["experiment_name"],
    group="development_catboost_cv",
    job_type="cross_validation",
    config=catboost_cv_config,
    mode="online",
) as run:

    def log_catboost_fold(result):
        run.log({
            f"cv_fold/{key}": float(value)
            for key, value in result.items()
        })

    fold_results = training_module.cross_validate_model(
        X=X_cv,
        y=y_cv,
        cv_splits=cv_splits,
        config=catboost_cv_config,
        on_fold_end=log_catboost_fold,
    )

    catboost_all_features_results = pd.DataFrame(fold_results)

    catboost_all_features_summary = (
        catboost_all_features_results[metric_columns]
        .agg(["mean", "std", "min", "max"])
        .T
    )
    catboost_all_features_summary.index.name = "metric"

    run.log({
        "cv/fold_results": wandb.Table(
            dataframe=catboost_all_features_results
        ),
        "cv/summary": wandb.Table(
            dataframe=catboost_all_features_summary.reset_index()
        ),
    })

    for metric in metric_columns:
        for statistic in ["mean", "std"]:
            run.summary[f"cv_{statistic}/{metric}"] = float(
                catboost_all_features_summary.loc[metric, statistic]
            )

    report_dir = PROJECT_ROOT / "outputs" / run.id
    report_dir.mkdir(parents=True, exist_ok=True)

    results_path = report_dir / "cv_fold_results.csv"
    summary_path = report_dir / "cv_summary.csv"

    catboost_all_features_results.to_csv(results_path, index=False)
    catboost_all_features_summary.to_csv(summary_path)

    artifact = wandb.Artifact(
        name="catboost-all-features-cv-report",
        type="evaluation",
    )
    artifact.add_file(str(results_path))
    artifact.add_file(str(summary_path))
    artifact.add_file(str(catboost_config_path))
    run.log_artifact(artifact)

    catboost_all_features_run_id = run.id

    print("Link W&B:", run.url)

display(catboost_all_features_summary.round(4))

Fold 1/5 | F1: 0.6654
Fold 2/5 | F1: 0.6843
Fold 3/5 | F1: 0.7068
Fold 4/5 | F1: 0.6903
Fold 5/5 | F1: 0.7195


Traceback (most recent call last):
  File "C:\Users\elsae\AppData\Local\Temp\ipykernel_30088\149891645.py", line 47, in <module>
    catboost_all_features_results[metric_columns]
  File "c:\Users\elsae\miniconda3\envs\tugasakhir\lib\site-packages\pandas\core\frame.py", line 4108, in __getitem__
    indexer = self.columns._get_indexer_strict(key, "columns")[1]
  File "c:\Users\elsae\miniconda3\envs\tugasakhir\lib\site-packages\pandas\core\indexes\base.py", line 6200, in _get_indexer_strict
    self._raise_if_missing(keyarr, indexer, axis_name)
  File "c:\Users\elsae\miniconda3\envs\tugasakhir\lib\site-packages\pandas\core\indexes\base.py", line 6249, in _raise_if_missing
    raise KeyError(f"None of [{key}] are in the [{axis_name}]")
KeyError: "None of [Index(['precision', 'recall', 'f1', 'average_precision', 'roc_auc'], dtype='object')] are in the [columns]"


cv_fold/average_precision,▁▄█▆▇
cv_fold/f1,▁▃▆▄█
cv_fold/fold,▁▃▅▆█
cv_fold/precision,▁▃█▃▇
cv_fold/recall,▂▅▁▇█
cv_fold/roc_auc,▁▅█▇▇
cv_fold/average_precision,0.76464
cv_fold/f1,0.71954
cv_fold/fold,5
cv_fold/precision,0.66429
cv_fold/recall,0.78481


KeyError: "None of [Index(['precision', 'recall', 'f1', 'average_precision', 'roc_auc'], dtype='object')] are in the [columns]"

In [26]:
import inspect

# Menyimpan ID run untuk pemulihan hasil di W&B.
catboost_recovery_run_id = run.id

print("Run ID:", catboost_recovery_run_id)
print("Tipe fold_results:", type(fold_results).__name__)
print("Isi hasil:", repr(fold_results)[:1500])
print("Ukuran tabel:", catboost_all_features_results.shape)
print("Kolom tabel:", catboost_all_features_results.columns.tolist())

print("\nFungsi cross_validate_model yang sedang digunakan:")
print(inspect.getsource(training_module.cross_validate_model))

Run ID: awr5x4m9
Tipe fold_results: NoneType
Isi hasil: None
Ukuran tabel: (0, 0)
Kolom tabel: []

Fungsi cross_validate_model yang sedang digunakan:
def cross_validate_model(
    X,
    y,
    cv_splits,
    config,
    on_fold_end=None,
):
    """Melatih pipeline baru dan mengevaluasinya pada setiap fold."""

    from src.evaluation import evaluate_model

    fold_results = []

    for fold, (train_idx, valid_idx) in enumerate(
        cv_splits,
        start=1,
    ):
        # Preprocessing dan model dilatih hanya pada training fold.
        pipeline = train_model(
            X.iloc[train_idx],
            y.iloc[train_idx],
            config,
        )

        metrics, _, _ = evaluate_model(
            pipeline,
            X.iloc[valid_idx],
            y.iloc[valid_idx],
            threshold=config["evaluation"]["threshold"],
        )

        result = {
            "fold": fold,
            **metrics,
        }

        fold_results.append(result)

        print(
       

In [27]:
# Memuat perbaikan fungsi untuk pemakaian berikutnya.
importlib.reload(training_module)

saved_catboost_run = wandb.Api().run(
    "elsaelisa/ecommerce-purchase-prediction/awr5x4m9"
)

history_keys = ["cv_fold/fold"] + [
    f"cv_fold/{metric}"
    for metric in metric_columns
]

history_rows = list(
    saved_catboost_run.scan_history(keys=history_keys)
)

if not history_rows:
    raise ValueError(
        "Riwayat fold belum ditemukan. Kirim pesan ini; "
        "jangan training ulang dulu."
    )

catboost_all_features_results = (
    pd.DataFrame(history_rows)[history_keys]
    .rename(columns={
        key: key.removeprefix("cv_fold/")
        for key in history_keys
    })
)

catboost_all_features_results["fold"] = (
    catboost_all_features_results["fold"].astype(int)
)

catboost_all_features_results = (
    catboost_all_features_results
    .sort_values("fold")
    .reset_index(drop=True)
)

assert len(catboost_all_features_results) == 5
assert set(catboost_all_features_results["fold"]) == {1, 2, 3, 4, 5}

catboost_all_features_summary = (
    catboost_all_features_results[metric_columns]
    .agg(["mean", "std", "min", "max"])
    .T
)

catboost_all_features_summary.index.name = "metric"
catboost_all_features_run_id = "awr5x4m9"

display(catboost_all_features_summary.round(4))

,mean,std,min,max
metric,,,,
precision,0.6340,0.0323,0.5973,0.6718
recall,0.7658,0.0173,0.7458,0.7848
f1,0.6933,0.0208,0.6654,0.7195
average_precision,0.7448,0.0333,0.6963,0.7795
roc_auc,0.9305,0.0094,0.9153,0.9386


In [28]:
with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    id=catboost_all_features_run_id,
    resume="must",
    mode="online",
) as run:

    run.log({
        "cv/fold_results": wandb.Table(
            dataframe=catboost_all_features_results
        ),
        "cv/summary": wandb.Table(
            dataframe=catboost_all_features_summary.reset_index()
        ),
    })

    for metric in metric_columns:
        for statistic in ["mean", "std"]:
            run.summary[f"cv_{statistic}/{metric}"] = float(
                catboost_all_features_summary.loc[metric, statistic]
            )

    run.summary["recovery/source"] = "logged_fold_history"
    run.summary["recovery/completed_folds"] = 5

    report_dir = PROJECT_ROOT / "outputs" / run.id
    report_dir.mkdir(parents=True, exist_ok=True)

    results_path = report_dir / "cv_fold_results.csv"
    summary_path = report_dir / "cv_summary.csv"

    catboost_all_features_results.to_csv(results_path, index=False)
    catboost_all_features_summary.to_csv(summary_path)

    artifact = wandb.Artifact(
        name="catboost-all-features-cv-report",
        type="evaluation",
    )
    artifact.add_file(str(results_path))
    artifact.add_file(str(summary_path))
    artifact.add_file(str(catboost_config_path))
    run.log_artifact(artifact)

    print("Ringkasan dan laporan dilengkapi:", run.url)

Ringkasan dan laporan dilengkapi: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/awr5x4m9


cv_fold/average_precision,0.76464
cv_fold/f1,0.71954
cv_fold/fold,5
cv_fold/precision,0.66429
cv_fold/recall,0.78481
cv_fold/roc_auc,0.93573
cv_mean/average_precision,0.74479
cv_mean/f1,0.69328
cv_mean/precision,0.63397
cv_mean/recall,0.76583
+8,...


In [29]:
catboost_without_pagevalues_config = deepcopy(
    catboost_cv_config
)

catboost_without_pagevalues_config["experiment_name"] = (
    "catboost_without_pagevalues_cv"
)

catboost_without_pagevalues_config["features"][
    "exclude_columns"
] = ["PageValues"]

catboost_without_pagevalues_config["feature_columns"] = (
    X_without_pagevalues.columns.tolist()
)

catboost_without_pagevalues_config_path = (
    PROJECT_ROOT / "configs" / "catboost_without_pagevalues.yaml"
)

with catboost_without_pagevalues_config_path.open(
    "w", encoding="utf-8"
) as file:
    yaml.safe_dump(
        catboost_without_pagevalues_config,
        file,
        sort_keys=False,
    )

print("Fitur:", X_without_pagevalues.shape)
print(
    "Threshold:",
    catboost_without_pagevalues_config["evaluation"]["threshold"],
)
print(
    "Fitur yang dihapus:",
    catboost_without_pagevalues_config["features"]["exclude_columns"],
)

Fitur: (7406, 16)
Threshold: 0.3
Fitur yang dihapus: ['PageValues']


In [30]:
with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    name=catboost_without_pagevalues_config["experiment_name"],
    group="development_catboost_cv",
    job_type="cross_validation",
    config=catboost_without_pagevalues_config,
    mode="online",
) as run:

    catboost_without_pagevalues_run_id = run.id

    def log_catboost_without_pagevalues_fold(result):
        run.log({
            f"cv_fold/{key}": float(value)
            for key, value in result.items()
        })

    fold_results = training_module.cross_validate_model(
        X=X_without_pagevalues,
        y=y_cv,
        cv_splits=cv_splits,
        config=catboost_without_pagevalues_config,
        on_fold_end=log_catboost_without_pagevalues_fold,
    )

    catboost_without_pagevalues_results = pd.DataFrame(
        fold_results
    )

    catboost_without_pagevalues_summary = (
        catboost_without_pagevalues_results[metric_columns]
        .agg(["mean", "std", "min", "max"])
        .T
    )
    catboost_without_pagevalues_summary.index.name = "metric"

    run.log({
        "cv/fold_results": wandb.Table(
            dataframe=catboost_without_pagevalues_results
        ),
        "cv/summary": wandb.Table(
            dataframe=catboost_without_pagevalues_summary.reset_index()
        ),
    })

    for metric in metric_columns:
        for statistic in ["mean", "std"]:
            run.summary[f"cv_{statistic}/{metric}"] = float(
                catboost_without_pagevalues_summary.loc[
                    metric, statistic
                ]
            )

    report_dir = PROJECT_ROOT / "outputs" / run.id
    report_dir.mkdir(parents=True, exist_ok=True)

    results_path = report_dir / "cv_fold_results.csv"
    summary_path = report_dir / "cv_summary.csv"

    catboost_without_pagevalues_results.to_csv(
        results_path,
        index=False,
    )
    catboost_without_pagevalues_summary.to_csv(summary_path)

    artifact = wandb.Artifact(
        name="catboost-without-pagevalues-cv-report",
        type="evaluation",
    )
    artifact.add_file(str(results_path))
    artifact.add_file(str(summary_path))
    artifact.add_file(str(catboost_without_pagevalues_config_path))
    run.log_artifact(artifact)

    print("Link W&B:", run.url)

display(catboost_without_pagevalues_summary.round(4))

Fold 1/5 | F1: 0.4088
Fold 2/5 | F1: 0.4606
Fold 3/5 | F1: 0.3983
Fold 4/5 | F1: 0.3956
Fold 5/5 | F1: 0.4208
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/69ygo1i8


cv_fold/average_precision,▃▁▅██
cv_fold/f1,▂█▁▁▄
cv_fold/fold,▁▃▅▆█
cv_fold/precision,▁█▂▄▅
cv_fold/recall,▆█▃▁▄
cv_fold/roc_auc,▁▁█▅▂
cv_fold/average_precision,0.39932
cv_fold/f1,0.42082
cv_fold/fold,5
cv_fold/precision,0.43304
cv_fold/recall,0.40928


,mean,std,min,max
metric,,,,
precision,0.4213,0.0287,0.3893,0.4635
recall,0.4134,0.0319,0.3755,0.4576
f1,0.4168,0.0264,0.3956,0.4606
average_precision,0.3823,0.0173,0.3599,0.3993
roc_auc,0.7835,0.0097,0.7757,0.7977


In [31]:
import shutil

active_config_path = (
    PROJECT_ROOT / "configs" / "experiment.yaml"
)

shutil.copy2(
    PROJECT_ROOT / "configs" / "catboost_all_features.yaml",
    active_config_path,
)

with active_config_path.open("r", encoding="utf-8") as file:
    active_config = yaml.safe_load(file)

print("Eksperimen:", active_config["experiment_name"])
print("Model:", active_config["model"]["name"])
print("Threshold:", active_config["evaluation"]["threshold"])

Eksperimen: catboost_all_features
Model: catboost
Threshold: 0.3


In [32]:
source_config_path = (
    PROJECT_ROOT / "configs" / "catboost_without_pagevalues.yaml"
)

with source_config_path.open("r", encoding="utf-8") as file:
    catboost_without_final_config = yaml.safe_load(file)

# Menyiapkan konfigurasi untuk training seluruh bagian training.
catboost_without_final_config["experiment_name"] = (
    "catboost_without_pagevalues"
)
catboost_without_final_config.pop("cv", None)

with active_config_path.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        catboost_without_final_config,
        file,
        sort_keys=False,
    )

print("Eksperimen:", catboost_without_final_config["experiment_name"])
print(
    "Fitur yang dihapus:",
    catboost_without_final_config["features"]["exclude_columns"],
)

Eksperimen: catboost_without_pagevalues
Fitur yang dihapus: ['PageValues']


In [33]:
project_path = "elsaelisa/ecommerce-purchase-prediction"

comparison_specs = [
    (
        "XGBoost tuned", "all_features", "22cse45y",
        comparison_results.iloc[0]["run_id"],
    ),
    (
        "XGBoost tuned", "without_pagevalues", "zruv9kiq",
        without_pagevalues_comparison.iloc[0]["run_id"],
    ),
    (
        "CatBoost baseline", "all_features", "wxyu3tcu",
        catboost_all_features_run_id,
    ),
    (
        "CatBoost baseline", "without_pagevalues", "pz7e2gh4",
        catboost_without_pagevalues_run_id,
    ),
]

api = wandb.Api()
model_comparison_records = []

for model_name, variant, training_id, cv_id in comparison_specs:
    training_run = api.run(f"{project_path}/{training_id}")
    cv_run = api.run(f"{project_path}/{cv_id}")

    record = {
        "model": model_name,
        "variant": variant,
        "threshold": float(
            training_run.config["evaluation"]["threshold"]
        ),
        "cv_mean_f1": float(cv_run.summary["cv_mean/f1"]),
        "cv_std_f1": float(cv_run.summary["cv_std/f1"]),
        "training_run_path": f"{project_path}/{training_id}",
        "cv_run_path": f"{project_path}/{cv_id}",
    }

    for metric in metric_columns:
        record[f"validation_{metric}"] = float(
            training_run.summary[f"validation/{metric}"]
        )

    model_comparison_records.append(record)

xgb_catboost_comparison = pd.DataFrame(
    model_comparison_records
)

display(
    xgb_catboost_comparison[[
        "model",
        "variant",
        "cv_mean_f1",
        "validation_f1",
        "validation_average_precision",
    ]].round(4)
)

,model,variant,cv_mean_f1,validation_f1,validation_average_precision
0,XGBoost tuned,all_features,0.6972,0.6559,0.7419
1,XGBoost tuned,without_pagevalues,0.4065,0.3503,0.3299
2,CatBoost baseline,all_features,0.6933,0.6485,0.7401
3,CatBoost baseline,without_pagevalues,0.4168,0.3714,0.3241


In [34]:
from importlib.metadata import version

# Memperbarui daftar dependency proyek.
requirements_path = PROJECT_ROOT / "requirements.txt"

requirements_lines = [
    line
    for line in requirements_path.read_text(
        encoding="utf-8"
    ).splitlines()
    if not line.strip().lower().startswith("catboost==")
]

requirements_lines.append(f"catboost=={version('catboost')}")

requirements_path.write_text(
    "\n".join(requirements_lines) + "\n",
    encoding="utf-8",
)

# Menyimpan laporan perbandingan ke W&B.
with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    name="xgboost_catboost_comparison",
    job_type="analysis",
    mode="online",
    config={
        "evaluation_split": "original_validation",
        "cv_data_split": "original_train",
        "threshold": 0.3,
        "training_run_paths": (
            xgb_catboost_comparison["training_run_path"].tolist()
        ),
        "cv_run_paths": (
            xgb_catboost_comparison["cv_run_path"].tolist()
        ),
    },
) as run:

    run.log({
        "comparison/results": wandb.Table(
            dataframe=xgb_catboost_comparison
        )
    })

    report_dir = PROJECT_ROOT / "outputs" / run.id
    report_dir.mkdir(parents=True, exist_ok=True)

    report_path = report_dir / "xgboost_catboost_comparison.csv"
    xgb_catboost_comparison.to_csv(report_path, index=False)

    artifact = wandb.Artifact(
        name="xgboost-catboost-comparison",
        type="evaluation",
    )

    artifact.add_file(str(report_path))
    artifact.add_file(str(requirements_path))

    for source_file in (PROJECT_ROOT / "src").rglob("*.py"):
        artifact.add_file(
            str(source_file),
            name=source_file.relative_to(PROJECT_ROOT).as_posix(),
        )

    run.log_artifact(artifact)

    print("Laporan tersimpan:", report_path)
    print("Link W&B:", run.url)

Laporan tersimpan: C:\Users\elsae\Downloads\ecommerce-purchase-prediction\outputs\ix3m010s\xgboost_catboost_comparison.csv
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/ix3m010s


In [35]:
# Membaca konfigurasi XGBoost terbaik dari eksperimen sebelumnya.
source_path = (
    PROJECT_ROOT / "configs" / "xgb_tuned_all_features.yaml"
)

with source_path.open("r", encoding="utf-8") as file:
    weighted_config = yaml.safe_load(file)

weighted_config["experiment_name"] = (
    "xgb_all_features_auto_weight"
)

weighted_config["resampling"] = {"name": "none"}

# Penanda khusus: kode training kita akan menghitung bobotnya.
weighted_config["model"]["params"]["scale_pos_weight"] = "auto"

weighted_config["selection"] = {
    "purpose": "class_weight_comparison",
    "base_config": "configs/xgb_tuned_all_features.yaml",
    "weight_calculation": "training_partition_only",
}

weighted_config_path = (
    PROJECT_ROOT / "configs" / "xgb_all_features_auto_weight.yaml"
)

with weighted_config_path.open("w", encoding="utf-8") as file:
    yaml.safe_dump(weighted_config, file, sort_keys=False)

print("Eksperimen:", weighted_config["experiment_name"])
print("Model:", weighted_config["model"]["name"])
print(
    "Pembobotan:",
    weighted_config["model"]["params"]["scale_pos_weight"],
)
print("Threshold:", weighted_config["evaluation"]["threshold"])

Eksperimen: xgb_all_features_auto_weight
Model: xgboost
Pembobotan: auto
Threshold: 0.3


In [36]:
# Memuat perubahan terbaru pada src/training.py.
importlib.reload(training_module)

weight_records = []

for fold, (train_idx, valid_idx) in enumerate(
    cv_splits,
    start=1,
):
    y_fold_train = y_cv.iloc[train_idx]

    resolved_config = training_module.resolve_training_config(
        y_fold_train,
        weighted_config,
    )

    weight_records.append({
        "fold": fold,
        "training_sessions": len(train_idx),
        "not_purchased": int((y_fold_train == 0).sum()),
        "purchased": int((y_fold_train == 1).sum()),
        "scale_pos_weight": resolved_config[
            "model"
        ]["params"]["scale_pos_weight"],
    })

weight_summary = pd.DataFrame(weight_records)

display(weight_summary.round(4))

print(
    "Penanda konfigurasi awal:",
    weighted_config["model"]["params"]["scale_pos_weight"],
)

,fold,training_sessions,not_purchased,purchased,scale_pos_weight
0,1,5924,4978,946,5.2622
1,2,5925,4978,947,5.2566
2,3,5925,4978,947,5.2566
3,4,5925,4979,946,5.2632
4,5,5925,4979,946,5.2632


Penanda konfigurasi awal: auto


In [37]:
weighted_cv_config = deepcopy(weighted_config)

weighted_cv_config["experiment_name"] = (
    "xgb_all_features_auto_weight_cv"
)

weighted_cv_config["cv"] = {
    "n_splits": 5,
    "splitter": "StratifiedGroupKFold",
    "shuffle": False,
    "data_split": "original_train",
}

with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    name=weighted_cv_config["experiment_name"],
    group="development_xgb_class_weight",
    job_type="cross_validation",
    config=weighted_cv_config,
    mode="online",
) as run:

    weighted_cv_run_id = run.id

    def log_weighted_fold(result):
        run.log({
            f"cv_fold/{key}": float(value)
            for key, value in result.items()
        })

    fold_results = training_module.cross_validate_model(
        X=X_cv,
        y=y_cv,
        cv_splits=cv_splits,
        config=weighted_cv_config,
        on_fold_end=log_weighted_fold,
    )

    weighted_cv_results = pd.DataFrame(fold_results)

    weighted_cv_summary = (
        weighted_cv_results[metric_columns]
        .agg(["mean", "std", "min", "max"])
        .T
    )
    weighted_cv_summary.index.name = "metric"

    run.log({
        "cv/fold_results": wandb.Table(
            dataframe=weighted_cv_results
        ),
        "cv/summary": wandb.Table(
            dataframe=weighted_cv_summary.reset_index()
        ),
    })

    for metric in metric_columns:
        for statistic in ["mean", "std"]:
            run.summary[f"cv_{statistic}/{metric}"] = float(
                weighted_cv_summary.loc[metric, statistic]
            )

    report_dir = PROJECT_ROOT / "outputs" / run.id
    report_dir.mkdir(parents=True, exist_ok=True)

    results_path = report_dir / "cv_fold_results.csv"
    summary_path = report_dir / "cv_summary.csv"

    weighted_cv_results.to_csv(results_path, index=False)
    weighted_cv_summary.to_csv(summary_path)

    artifact = wandb.Artifact(
        name="xgb-auto-weight-cv-report",
        type="evaluation",
    )
    artifact.add_file(str(results_path))
    artifact.add_file(str(summary_path))
    artifact.add_file(str(weighted_config_path))
    artifact.add_file(
        str(PROJECT_ROOT / "src" / "training.py"),
        name="src/training.py",
    )
    run.log_artifact(artifact)

    print("Link W&B:", run.url)

display(weighted_cv_summary.round(4))

Fold 1/5 | F1: 0.5699
Fold 2/5 | F1: 0.5793
Fold 3/5 | F1: 0.5922
Fold 4/5 | F1: 0.5962
Fold 5/5 | F1: 0.5964
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/sllsgwgz


cv_fold/average_precision,▂▁█▆▆
cv_fold/f1,▁▃▇██
cv_fold/fold,▁▃▅▆█
cv_fold/precision,▁▁▅▇█
cv_fold/recall,▁█▇▇▃
cv_fold/roc_auc,▁▂█▆▆
cv_fold/scale_pos_weight,▇▁▁██
cv_fold/average_precision,0.76353
cv_fold/f1,0.59639
cv_fold/fold,5
cv_fold/precision,0.44421


,mean,std,min,max
metric,,,,
precision,0.4310,0.0117,0.4181,0.4442
recall,0.9197,0.0180,0.8945,0.9364
f1,0.5868,0.0117,0.5699,0.5964
average_precision,0.7461,0.0343,0.7041,0.7851
roc_auc,0.9333,0.0071,0.9241,0.9412


In [ ]:
# Mengaktifkan konfigurasi XGBoost dengan bobot otomatis.
shutil.copy2(
    weighted_config_path,
    active_config_path,
)

with active_config_path.open("r", encoding="utf-8") as file:
    active_config = yaml.safe_load(file)

print("Eksperimen:", active_config["experiment_name"])
print(
    "Pembobotan:",
    active_config["model"]["params"]["scale_pos_weight"],
)                   

Eksperimen: xgb_all_features_auto_weight
Pembobotan: auto


In [39]:
import joblib
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)

weighted_saved_run = wandb.Api().run(
    "elsaelisa/ecommerce-purchase-prediction/d9n08p0g"
)
weighted_saved_config = dict(weighted_saved_run.config)

# Mengambil validation awal.
weighted_X_val, weighted_y_val = get_split(
    df,
    manifest,
    split_name="validation",
    target_column=weighted_saved_config["data"]["target"],
    exclude_columns=weighted_saved_config["features"]["exclude_columns"],
)

# Memuat pipeline yang sudah dilatih.
weighted_pipeline = joblib.load(
    PROJECT_ROOT
    / weighted_saved_config["output_dir"]
    / weighted_saved_run.id
    / "pipeline.joblib"
)

weighted_actual_scale_pos_weight = float(
    weighted_pipeline.named_steps["model"].get_params()[
        "scale_pos_weight"
    ]
)

# Menghitung skor sekali untuk seluruh sesi validation.
weighted_val_probabilities = weighted_pipeline.predict_proba(
    weighted_X_val
)[:, 1]

weighted_threshold_records = []

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    predictions = (
        weighted_val_probabilities >= threshold
    ).astype(int)

    weighted_threshold_records.append({
        "threshold": threshold,
        "precision": precision_score(
            weighted_y_val, predictions, zero_division=0
        ),
        "recall": recall_score(
            weighted_y_val, predictions, zero_division=0
        ),
        "f1": f1_score(
            weighted_y_val, predictions, zero_division=0
        ),
        "predicted_purchases": int(predictions.sum()),
    })

weighted_threshold_results = pd.DataFrame(
    weighted_threshold_records
)

print("Bobot aktual model:", weighted_actual_scale_pos_weight)
display(weighted_threshold_results.round(4))

Bobot aktual model: 5.260355029585799


,threshold,precision,recall,f1,predicted_purchases
0,0.3,0.4057,0.9297,0.5649,848
1,0.4,0.4761,0.8865,0.6195,689
2,0.5,0.5204,0.8270,0.6388,588
3,0.6,0.5514,0.7973,0.6519,535
4,0.7,0.6040,0.7378,0.6642,452
5,0.8,0.6508,0.6297,0.6401,358
6,0.9,0.7970,0.4351,0.5629,202


In [40]:
best_weighted_row = (
    weighted_threshold_results
    .sort_values(
        ["f1", "threshold"],
        ascending=[False, True],
    )
    .iloc[0]
)

weighted_selection = {
    "source_run_path": (
        "elsaelisa/ecommerce-purchase-prediction/d9n08p0g"
    ),
    "model_name": "xgboost",
    "feature_variant": "all_features",
    "threshold": float(best_weighted_row["threshold"]),
    "scale_pos_weight": weighted_actual_scale_pos_weight,
    "selection_metric": "f1",
    "selection_split": "validation",
    "tested_thresholds": (
        weighted_threshold_results["threshold"].tolist()
    ),
}

selection_path = (
    PROJECT_ROOT / "configs" / "weighted_candidate_selection.yaml"
)

with selection_path.open("w", encoding="utf-8") as file:
    yaml.safe_dump(weighted_selection, file, sort_keys=False)

with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    name="xgb_auto_weight_threshold_search",
    job_type="threshold_tuning",
    config=weighted_selection,
    mode="online",
) as run:

    run.log({
        "threshold_search/results": wandb.Table(
            dataframe=weighted_threshold_results
        )
    })

    run.summary["best_tested_threshold"] = float(
        best_weighted_row["threshold"]
    )

    for metric in ["precision", "recall", "f1"]:
        run.summary[f"selected_validation/{metric}"] = float(
            best_weighted_row[metric]
        )

    report_dir = PROJECT_ROOT / "outputs" / run.id
    report_dir.mkdir(parents=True, exist_ok=True)

    report_path = report_dir / "weighted_threshold_search.csv"
    weighted_threshold_results.to_csv(report_path, index=False)

    artifact = wandb.Artifact(
        name="xgb-auto-weight-threshold-report",
        type="evaluation",
    )
    artifact.add_file(str(report_path))
    artifact.add_file(str(selection_path))
    run.log_artifact(artifact)

    print("Threshold terpilih:", weighted_selection["threshold"])
    print("Konfigurasi kandidat:", selection_path)
    print("Link W&B:", run.url)

Threshold terpilih: 0.7
Konfigurasi kandidat: C:\Users\elsae\Downloads\ecommerce-purchase-prediction\configs\weighted_candidate_selection.yaml
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/83s5lg90


best_tested_threshold,0.7
selected_validation/f1,0.66423
selected_validation/precision,0.60398
selected_validation/recall,0.73784
